In [1]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


# 📥 Étape 1 : Acquisition des Données & Multi-Sources (Squelette Étudiant)

Cette étape correspond au premier chapitre du pipeline de Data Science. L'objectif est d'identifier, d'importer et de consolider vos jeux de données bruts issus de différentes sources (fichiers CSV locaux, requêtes API, bases de données, etc.).

### 1. Initialisation de l'environnement

In [2]:
import os
import sys
import pandas as pd
import numpy as np

print("📥 Acquisition des données démarrée")
BASE_DIR = os.path.abspath("..")
sys.path.append(BASE_DIR)

print("📦 Environment OK — prêt pour acquisition")

📥 Acquisition des données démarrée
📦 Environment OK — prêt pour acquisition


### 2. Chargement de la source de données principale

**À COMPLÉTER PAR L'ÉTUDIANT :**
Chargez votre jeu de données principal (par exemple un fichier CSV stocké dans `data/raw/`).

In [3]:
main_data_path = os.path.join(BASE_DIR, "data", "raw", "train.csv")

if not os.path.exists(main_data_path):
    raise FileNotFoundError(f"❌ Dataset introuvable : {main_data_path}")

df_main = pd.read_csv(main_data_path)

print("✔ Dataset chargé :", df_main.shape)
df_main.head()

print("\n📊 INFO DATASET")
print(df_main.info())

print("\n❌ Missing values (top 10)")
print(df_main.isnull().sum().sort_values(ascending=False).head(10))

✔ Dataset chargé : (1460, 81)

📊 INFO DATASET
<class 'pandas.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   str    
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   str    
 6   Alley          91 non-null     str    
 7   LotShape       1460 non-null   str    
 8   LandContour    1460 non-null   str    
 9   Utilities      1460 non-null   str    
 10  LotConfig      1460 non-null   str    
 11  LandSlope      1460 non-null   str    
 12  Neighborhood   1460 non-null   str    
 13  Condition1     1460 non-null   str    
 14  Condition2     1460 non-null   str    
 15  BldgType       1460 non-null   str    
 16  HouseStyle     1460 non-null   str    
 17  OverallQual    14

### 3. Intégration de données secondaires (Multi-Sources)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Mettez en place la récupération de vos données complémentaires (par exemple, appels d'API fictifs ou réels, données météo, géographiques, ou financiers complémentaires).

In [4]:
categories_info = pd.DataFrame({
    "Neighborhood": [
        "CollgCr", "Veenker", "Crawfor",
        "NoRidge", "Mitchel", "OldTown"
    ],
    "zone_type": [
        "Standard", "Premium", "Premium",
        "Luxury", "Standard", "Standard"
    ],
    "coef_multiplicateur": [
        1.0, 1.2, 1.3,
        1.5, 1.1, 0.95
    ]
})

print("✔ Données secondaires créées")
categories_info.head()

✔ Données secondaires créées


,Neighborhood,zone_type,coef_multiplicateur
0,CollgCr,Standard,1.0
1,Veenker,Premium,1.2
2,Crawfor,Premium,1.3
3,NoRidge,Luxury,1.5
4,Mitchel,Standard,1.1


### 4. Fusion des sources (Optionnel)

**À COMPLÉTER PAR L'ÉTUDIANT :**
Associez vos différentes sources de données en utilisant des jointures (`pd.merge`) pertinentes.

In [5]:
df_main["Neighborhood"] = df_main["Neighborhood"].astype(str).str.strip()
categories_info["Neighborhood"] = categories_info["Neighborhood"].astype(str).str.strip()

df_merged = df_main.merge(categories_info, on="Neighborhood", how="left")

print("✔ Fusion terminée :", df_merged.shape)
df_merged.head()

✔ Fusion terminée : (1460, 83)


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,zone_type,coef_multiplicateur
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,NaN,NaN,0,2,2008,WD,Normal,208500,Standard,1.0
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,NaN,NaN,0,5,2007,WD,Normal,181500,Premium,1.2
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,NaN,NaN,0,9,2008,WD,Normal,223500,Standard,1.0
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,NaN,NaN,0,2,2006,WD,Abnorml,140000,Premium,1.3
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,NaN,NaN,0,12,2008,WD,Normal,250000,Luxury,1.5


### 5. Consignation des données d'entrée brutes

Sauvegardez l'état brut de vos données d'entrée pour la suite du pipeline.

In [6]:
print("\n❌ Missing après merge")
print(df_merged[["zone_type", "coef_multiplicateur"]].isnull().sum())

# IMPORTANT FIX (évite ton problème 1158 NaN)
df_merged["zone_type"] = df_merged["zone_type"].fillna("Unknown")
df_merged["coef_multiplicateur"] = df_merged["coef_multiplicateur"].fillna(1.0)

output_path = os.path.join(BASE_DIR, "data", "processed", "raw_merged.csv")

os.makedirs(os.path.dirname(output_path), exist_ok=True)

df_merged.to_csv(output_path, index=False)

print("💾 Dataset sauvegardé :", output_path)


❌ Missing après merge
zone_type              1045
coef_multiplicateur    1045
dtype: int64
💾 Dataset sauvegardé : /Users/hobb/Documents/GitHub/aptispace-datascience-projet/data/processed/raw_merged.csv
